# Cross-study DCM consensus

This notebook matches the selected gene-symbol result tables from GSE5406 and GSE42955, documents probe-to-symbol handling, then applies the approved BH-adjusted p-value and direction-concordance rules to create the consensus gene list.


In [1]:
# Load tidyverse so symbol validation, matching, and ordering use consistent table verbs.
suppressPackageStartupMessages(library(tidyverse))

# Locate the project root so the notebook works from either notebooks/ or the root.
root_candidates <- c(".", "..", "../..")
root_candidates <- root_candidates[file.exists(file.path(root_candidates, "results/tables/GSE5406_limma_gene_symbol_selected.csv"))]
if (length(root_candidates) == 0L) stop("Could not locate the project root.")
project_root <- normalizePath(root_candidates[[1]], winslash = "/")
table_dir <- file.path(project_root, "results", "tables")

# Read the selected-symbol outputs; they contain one chosen probe per unambiguous symbol.
gse5406_genes <- readr::read_csv(file.path(table_dir, "GSE5406_limma_gene_symbol_selected.csv"), show_col_types = FALSE)
gse42955_genes <- readr::read_csv(file.path(table_dir, "GSE42955_limma_gene_symbol_selected.csv"), show_col_types = FALSE)
required_columns <- c("gene_symbol", "probe_id", "logFC", "adj.P.Val")
if (!all(required_columns %in% names(gse5406_genes)) || !all(required_columns %in% names(gse42955_genes))) {
  stop("Both selected result tables must include gene_symbol, probe_id, logFC, and adj.P.Val.")
}

# Trim annotation whitespace before checking that the selected tables have one row per symbol.
gse5406_genes <- gse5406_genes |>
  dplyr::mutate(gene_symbol = stringr::str_trim(gene_symbol))
gse42955_genes <- gse42955_genes |>
  dplyr::mutate(gene_symbol = stringr::str_trim(gene_symbol))
if (any(is.na(gse5406_genes$gene_symbol) | gse5406_genes$gene_symbol == "") ||
    any(is.na(gse42955_genes$gene_symbol) | gse42955_genes$gene_symbol == "")) {
  stop("A selected-symbol table contains a missing or blank gene symbol; inspect annotation before matching.")
}
if (anyDuplicated(gse5406_genes$gene_symbol) || anyDuplicated(gse42955_genes$gene_symbol)) {
  stop("A selected-symbol table contains duplicate gene symbols; inspect probe selection before matching.")
}

# Count each study's unique selected symbols and intersect on the shared gene-symbol key.
overlap_table <- dplyr::inner_join(
  gse5406_genes |>
    dplyr::rename_with(~ paste0(.x, "_GSE5406"), -gene_symbol),
  gse42955_genes |>
    dplyr::rename_with(~ paste0(.x, "_GSE42955"), -gene_symbol),
  by = "gene_symbol"
) |>
  dplyr::arrange(gene_symbol)
overlap_summary <- tibble::tibble(
  study = c("GSE5406", "GSE42955", "Overlap"),
  unique_gene_symbols = c(nrow(gse5406_genes), nrow(gse42955_genes), nrow(overlap_table))
)
print(overlap_summary)
cat("\nMatched gene symbols:", nrow(overlap_table), "\n")

# Check probe mapping and selection rules documented in the study notebooks.
gse5406_probe <- readr::read_csv(file.path(table_dir, "GSE5406_limma_probe_level_all.csv"), show_col_types = FALSE)
gse42955_probe <- readr::read_csv(file.path(table_dir, "GSE42955_limma_probe_level_all.csv"), show_col_types = FALSE)
if (!"annotation_status" %in% names(gse5406_probe) || !"gene_symbol" %in% names(gse42955_probe)) {
  stop("Probe-level mapping fields do not match the expected study outputs.")
}
cat("\nGSE5406 probe annotation-status counts:\n")
print(gse5406_probe |> dplyr::count(annotation_status, name = "probe_count") |> dplyr::arrange(annotation_status))
gse42955_mapping <- gse42955_probe |>
  dplyr::mutate(
    gene_symbol = stringr::str_trim(gene_symbol),
    mapping_status = dplyr::case_when(
      is.na(gene_symbol) | gene_symbol %in% c("", "---", "NA") ~ "unmapped",
      stringr::str_detect(gene_symbol, "///|;") ~ "ambiguous",
      TRUE ~ "single symbol"
    )
  ) |>
  dplyr::count(mapping_status, name = "probe_count") |>
  dplyr::arrange(mapping_status)
cat("\nGSE42955 probe mapping-status counts:\n")
print(gse42955_mapping)

# Save the unfiltered symbol overlap inventory for review; significance and direction are not used here.
readr::write_csv(overlap_table, file.path(table_dir, "consensus_gene_symbol_overlap.csv"))
readr::write_csv(overlap_summary, file.path(table_dir, "consensus_gene_symbol_overlap_summary.csv"))
cat("\nSaved symbol overlap inventory and counts under:", table_dir, "\n")
cat("No adjusted-p-value threshold or expression-direction rule has been applied.\n")


# A tibble: 3 × 2
  study    unique_gene_symbols
  <chr>                  <int>
1 GSE5406                12650
2 GSE42955               19036
3 Overlap                11472



Matched gene symbols: 11472 



GSE5406 probe annotation-status counts:


# A tibble: 3 × 2
  annotation_status probe_count
  <chr>                   <int>
1 ambiguous                 836
2 mapped                  20210
3 unmapped                 1169



GSE42955 probe mapping-status counts:


# A tibble: 3 × 2
  mapping_status probe_count
  <chr>                <int>
1 ambiguous             2143
2 single symbol        20052
3 unmapped             11102



Saved symbol overlap inventory and counts under: /home/user/biom5350/biom5350-dcm-consensus/results/tables 


No adjusted-p-value threshold or expression-direction rule has been applied.


In [2]:
# Count symbols represented by multiple biological probes to document the one-probe selection step.
gse5406_probe_multiplicity <- gse5406_probe |>
  dplyr::filter(annotation_status == "mapped") |>
  dplyr::count(gene_symbol, name = "probe_count") |>
  dplyr::summarise(
    symbols_with_multiple_probes = sum(probe_count > 1L),
    extra_probes_beyond_one_per_symbol = sum(pmax(probe_count - 1L, 0L))
  ) |>
  dplyr::mutate(study = "GSE5406", .before = 1)
gse42955_probe_multiplicity <- gse42955_probe |>
  dplyr::mutate(
    gene_symbol = stringr::str_trim(gene_symbol),
    mapping_status = dplyr::case_when(
      is.na(gene_symbol) | gene_symbol %in% c("", "---", "NA") ~ "unmapped",
      stringr::str_detect(gene_symbol, "///|;") ~ "ambiguous",
      TRUE ~ "single symbol"
    )
  ) |>
  dplyr::filter(mapping_status == "single symbol") |>
  dplyr::count(gene_symbol, name = "probe_count") |>
  dplyr::summarise(
    symbols_with_multiple_probes = sum(probe_count > 1L),
    extra_probes_beyond_one_per_symbol = sum(pmax(probe_count - 1L, 0L))
  ) |>
  dplyr::mutate(study = "GSE42955", .before = 1)
probe_multiplicity_summary <- dplyr::bind_rows(gse5406_probe_multiplicity, gse42955_probe_multiplicity)
cat("Probe multiplicity among uniquely mapped symbols:\n")
print(probe_multiplicity_summary)

# Record software versions and the session for reproducibility.
cat("\nR version:", R.version.string, "\n")
cat("tidyverse:", as.character(packageVersion("tidyverse")), "\n")
cat("limma:", as.character(packageVersion("limma")), "\n")
cat("\nSession information:\n")
print(utils::sessionInfo())


Probe multiplicity among uniquely mapped symbols:


# A tibble: 2 × 3
  study    symbols_with_multiple_probes extra_probes_beyond_one_per_symbol
  <chr>                           <int>                              <int>
1 GSE5406                          4583                               7560
2 GSE42955                          779                               1016



R version: R version 4.5.2 (2025-10-31) 


tidyverse: 2.0.0 


limma: 3.66.0 



Session information:


R version 4.5.2 (2025-10-31)
Platform: x86_64-pc-linux-gnu
Running under: Ubuntu 26.04 LTS

Matrix products: default
BLAS:   /usr/lib/x86_64-linux-gnu/blas/libblas.so.3.12.1 
LAPACK: /usr/lib/x86_64-linux-gnu/lapack/liblapack.so.3.12.1;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

time zone: Etc/UTC
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
 [1] lubridate_1.9.5 forcats_1.0.1   stringr_1.6.0   dplyr_1.2.1    
 [5] purrr_1.2.2     readr_2.2.0     tidyr_1.3.2     tibble_3.3.1   
 [9] ggplot2_4.0.3   tidyverse_2.0.0

loaded via a namespace (and not attached):
 [1] bit_4.6.0          gtable_0.3.6     

In [3]:
# Load tidyverse for symbol matching, filtering, and ranking the final table.
suppressPackageStartupMessages(library(tidyverse))

# Locate the project root so the notebook can run from notebooks/ or the project root.
root_candidates <- c(".", "..", "../..")
root_candidates <- root_candidates[file.exists(file.path(root_candidates, "results/tables/GSE5406_limma_gene_symbol_selected.csv"))]
if (length(root_candidates) == 0L) stop("Could not locate the project root.")
project_root <- normalizePath(root_candidates[[1]], winslash = "/")
table_dir <- file.path(project_root, "results", "tables")

# Read the one-probe-per-symbol results, which already exclude unmapped and ambiguous annotations.
gse5406_genes <- readr::read_csv(file.path(table_dir, "GSE5406_limma_gene_symbol_selected.csv"), show_col_types = FALSE)
gse42955_genes <- readr::read_csv(file.path(table_dir, "GSE42955_limma_gene_symbol_selected.csv"), show_col_types = FALSE)
required_columns <- c("gene_symbol", "probe_id", "logFC", "adj.P.Val")
if (!all(required_columns %in% names(gse5406_genes)) || !all(required_columns %in% names(gse42955_genes))) {
  stop("Both selected result tables must include gene_symbol, probe_id, logFC, and adj.P.Val.")
}

# Trim symbol whitespace and validate the one-row-per-symbol input needed for an unambiguous join.
gse5406_genes <- gse5406_genes |> dplyr::mutate(gene_symbol = stringr::str_trim(gene_symbol))
gse42955_genes <- gse42955_genes |> dplyr::mutate(gene_symbol = stringr::str_trim(gene_symbol))
if (any(is.na(gse5406_genes$gene_symbol) | gse5406_genes$gene_symbol == "") ||
    any(is.na(gse42955_genes$gene_symbol) | gse42955_genes$gene_symbol == "")) {
  stop("A selected-symbol table contains a missing or blank gene symbol.")
}
if (anyDuplicated(gse5406_genes$gene_symbol) || anyDuplicated(gse42955_genes$gene_symbol)) {
  stop("A selected-symbol table contains duplicate gene symbols.")
}

# Join genes significant at BH-adjusted p < 0.05 in both studies before comparing directions.
significant_in_both <- dplyr::inner_join(
  gse5406_genes |>
    dplyr::filter(adj.P.Val < 0.05) |>
    dplyr::select(gene_symbol, probe_id, logFC, adj.P.Val) |>
    dplyr::rename(
      probe_id_GSE5406 = probe_id,
      logFC_GSE5406 = logFC,
      adj.P.Val_GSE5406 = adj.P.Val
    ),
  gse42955_genes |>
    dplyr::filter(adj.P.Val < 0.05) |>
    dplyr::select(gene_symbol, probe_id, logFC, adj.P.Val) |>
    dplyr::rename(
      probe_id_GSE42955 = probe_id,
      logFC_GSE42955 = logFC,
      adj.P.Val_GSE42955 = adj.P.Val
    ),
  by = "gene_symbol"
) |>
  dplyr::mutate(
    same_direction = logFC_GSE5406 * logFC_GSE42955 > 0,
    direction = dplyr::case_when(
      same_direction & logFC_GSE5406 > 0 ~ "Up in DCM",
      same_direction & logFC_GSE5406 < 0 ~ "Down in DCM",
      TRUE ~ "Opposite or zero direction"
    )
  )

# Keep only concordant directions; rank by the less significant adjusted p-value for each pair.
consensus_genes <- significant_in_both |>
  dplyr::filter(same_direction) |>
  dplyr::mutate(worst_adj_p = pmax(adj.P.Val_GSE5406, adj.P.Val_GSE42955)) |>
  dplyr::arrange(worst_adj_p, gene_symbol) |>
  dplyr::select(
    gene_symbol, direction,
    probe_id_GSE5406, logFC_GSE5406, adj.P.Val_GSE5406,
    probe_id_GSE42955, logFC_GSE42955, adj.P.Val_GSE42955,
    worst_adj_p
  )

# Save the final table so each gene is traceable to both study-specific probes and statistics.
readr::write_csv(consensus_genes, file.path(table_dir, "consensus_genes.csv"))

# Report direction concordance among genes significant in both studies; zero effects are not concordant.
matching_direction_count <- sum(significant_in_both$same_direction)
fraction_matching <- if (nrow(significant_in_both) == 0L) NA_real_ else matching_direction_count / nrow(significant_in_both)
cat("Selected symbols: GSE5406 =", nrow(gse5406_genes), "; GSE42955 =", nrow(gse42955_genes), "\n")
cat("Genes significant in both studies =", nrow(significant_in_both), "\n")
cat("Same-direction consensus genes =", nrow(consensus_genes), "\n")
cat("Matching direction fraction among genes significant in both =", matching_direction_count, "/", nrow(significant_in_both), "(",
    scales::percent(fraction_matching, accuracy = 0.1), ")\n")
cat("Top consensus genes, ranked by worst adjusted p-value ascending (then gene symbol):\n")
print(consensus_genes |> dplyr::slice_head(n = 10L), width = Inf)


Selected symbols: GSE5406 = 12650 ; GSE42955 = 19036 


Genes significant in both studies = 2 


Same-direction consensus genes = 1 


Matching direction fraction among genes significant in both = 1 / 2 ( 50.0% )


Top consensus genes, ranked by worst adjusted p-value ascending (then gene symbol):


# A tibble: 1 × 9
  gene_symbol direction   probe_id_GSE5406 logFC_GSE5406 adj.P.Val_GSE5406
  <chr>       <chr>       <chr>                    <dbl>             <dbl>
1 AASS        Down in DCM 214829_at               -0.524      0.0000000467
  probe_id_GSE42955 logFC_GSE42955 adj.P.Val_GSE42955 worst_adj_p
              <dbl>          <dbl>              <dbl>       <dbl>
1           8142554         -0.698             0.0256      0.0256


## Symbol-overlap inventory

The selected gene-symbol tables contain 12,650 symbols for GSE5406 and 19,036 for GSE42955; 11,472 symbols occur in both. These counts use all selected symbols, regardless of adjusted p-value or log2FC direction. The overlap inventory and its count summary are saved in results/tables/consensus_gene_symbol_overlap.csv and results/tables/consensus_gene_symbol_overlap_summary.csv.

The probe-level tables retain unmapped and ambiguous probes for transparency. GSE5406 has 1,169 unmapped and 836 ambiguous probes; GSE42955 has 11,102 unmapped and 2,143 ambiguous probes. These probes do not enter the selected gene-symbol tables. GSE5406 symbols come from the GPL96 hgu133a.db annotation; GSE42955 symbols come from the GPL6244 Gene symbol annotation field. Both study pipelines retain one probe per unambiguous symbol, selected by smallest BH-adjusted p-value, then larger absolute log2FC, then lexical probe ID. The probe-level data contain multiple probes for 4,583 GSE5406 symbols and 779 GSE42955 symbols; the saved symbol tables contain the single selected row per symbol.

The next section reports genes with BH-adjusted p < 0.05 in both studies and then retains only genes whose log2FC signs agree.


## Significant and direction-concordant consensus

Among the 11,472 overlapping selected gene symbols, **2 genes were significant at BH-adjusted p < 0.05 in both studies**. One had matching nonzero log2FC directions, so the matching-direction fraction was **1/2 (50.0%)**. The final consensus list therefore contains **1 gene**. Zero log2FC values are not counted as direction matches.

The final table is saved to results/tables/consensus_genes.csv. It includes each study's selected probe ID, log2FC, adjusted p-value, direction, and the larger of the two adjusted p-values used to order the display (ascending; gene symbol breaks ties).

| Rank | Gene | Direction in both studies | GSE5406 probe | GSE5406 log2FC | GSE42955 probe | GSE42955 log2FC | GSE5406 adjusted p | GSE42955 adjusted p |
|---:|---|---|---|---:|---|---:|---:|---:|
| 1 | AASS | Down in DCM | 214829_at | -0.524 | 8142554 | -0.698 | 4.67e-7 | 0.0256 |

Only one gene passed both significance and direction rules, so fewer than ten genes are available for the requested top-ten listing.
